In [1]:
from builder import build_crm_input

In [2]:
import importlib
import builder
importlib.reload(builder)

from builder import build_crm_input

In [3]:
import inspect
print(inspect.getsource(build_crm_input))

def build_crm_input(row, tone_vector) -> CRMInput:
    return {
        "persona_id": row["persona_id"],
        "brand": row["brand"],
        "brand_tone_cluster": int(row["brand_tone_cluster"]),
        "avg_similarity": float(row["avg_similarity"]),
        # ingredient_affinity 제거 (이 CSV에는 없음)
        "tone_vector": tone_vector,
        "product_cnt": int(row["product_cnt"]),
    }



In [4]:
import pandas as pd
import pickle
from builder import build_crm_input

# 1. CSV 로드
df = pd.read_csv("../data_csv/persona_brand_tone_strategy.csv")
row = df.iloc[0]

# 2. tone_vectors 로드 (실제 존재하는 파일)
with open("../data_csv/tone_vectors.pkl", "rb") as f:
    tone_vectors = pickle.load(f)

# 3. brand_tone_cluster → tone_name 매핑 (논리 자산)
CLUSTER_TO_TONE = {
    0: "Scientific",
    1: "Emotional",
    2: "Luxury",
    3: "Casual"
}

cluster_id = int(row["brand_tone_cluster"])
tone_name = CLUSTER_TO_TONE[cluster_id]

# 4. tone_vector 선택
tone_vector = tone_vectors[tone_name]

# 5. crm_input 생성
crm_input = build_crm_input(row, tone_vector)

# 6. 설명용 톤 컨텍스트 (재분류 ❌)
CLUSTER_TO_POSITION = {
    0: "고기능/클리니컬 톤",
    1: "수분 중심 톤",
    2: "프리미엄/헤리티지 톤",
    3: "트렌디/컬러 중심 톤"
}

agent_input = {
    "crm_result": crm_input,
    "tone_context": {
        "brand_tone_cluster": cluster_id,
        "tone_name": tone_name,
        "brand_position": CLUSTER_TO_POSITION[cluster_id]
    }
}